In [ ]:
# 1. Dependencies Setup: Install required packages
print("Installing dependencies... This may take a few minutes.")

# Downgrade Numpy for compatibility and install training dependencies
!pip uninstall -y numpy
!pip install "numpy<2.0" ultralytics timm sahi grad-cam

import numpy as np
import ultralytics
print(f"Numpy Version: {np.__version__}")
print(f"Ultralytics Version: {ultralytics.__version__}")
print("\n[IMPORTANT] If Numpy is still 2.x, please click 'RESTART SESSION' in the Kaggle UI now!")

In [ ]:
# 2. Environment Setup: Define Paths and Source Code
from pathlib import Path
import sys, os

# --- Path Definitions ---
KAGGLE_DATASET = Path("/kaggle/input/datasets/mohamedtamzirt1")
SOURCE_ROOT = KAGGLE_DATASET / "weapon-detection-source"
DATA_ROOT = KAGGLE_DATASET / "wd-data/processed/yolo_dataset"
DATA_YAML = DATA_ROOT / "data.yaml"
WEIGHTS_DIR = Path("/kaggle/working/weights")
WEIGHTS_DIR.mkdir(exist_ok=True)

print(f"Source: {SOURCE_ROOT}")
print(f"Data: {DATA_ROOT}")

# --- Source Setup ---
if SOURCE_ROOT.exists():
    # Add source to sys.path so we can import modules
    if str(SOURCE_ROOT) not in sys.path:
        sys.path.append(str(SOURCE_ROOT))
    
    # IMPORTANT: We MUST stay in /kaggle/working because /kaggle/input is READ-ONLY.
    # Ultralytics will try to download the backbone weights (e.g. yolo11m.pt) to the CWD.
    os.chdir("/kaggle/working")
    print(f"✅ CWD set to writable directory: {os.getcwd()}")
    print(f"✅ Source code added to sys.path from: {SOURCE_ROOT}")
else:
    print(f"❌ ERROR: Source not found at {SOURCE_ROOT}")

In [ ]:
# 3. Imports
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torch.cuda.amp import GradScaler, autocast
from tqdm import tqdm
import yaml
from pathlib import Path

try:
    from models.hybrid_model import HybridWeaponDetector
    from ultralytics.data.dataset import YOLODataset
    from ultralytics.data.utils import check_det_dataset
    print("✅ Project modules imported successfully.")
except ImportError as e:
    print(f"[ERROR] Import failed: {e}")
    print("Check if 'weapon-detection-source' contains 'models/' and 'ultralytics' is installed.")

In [ ]:
def get_dataloaders(data_yaml_path, data_root, batch_size=32, imgsz=640):
    data_cfg = check_det_dataset(data_yaml_path)
    
    # --- Automatic Path Alignment ---
    for split in ['train', 'val', 'test']:
        if split in data_cfg:
            # Construct absolute path using discovered data_root
            data_cfg[split] = str(data_root / split / "images")
    # -------------------------------
    
    train_set = YOLODataset(img_path=data_cfg['train'], imgsz=imgsz, augment=True, batch_size=batch_size, task='detect', data=data_cfg)
    val_set = YOLODataset(img_path=data_cfg['val'], imgsz=imgsz, augment=False, batch_size=batch_size, task='detect', data=data_cfg)
    
    train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=True, collate_fn=train_set.collate_fn)
    val_loader = DataLoader(val_set, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True, collate_fn=val_set.collate_fn)
    
    return train_loader, val_loader

### 3. HybridTrainer (Dual GPU + AMP + Class Weighting)

In [ ]:
class HybridTrainer:
    def __init__(self, model, train_loader, val_loader, device="cuda", weights_dir=None):
        self.device = device
        self.weights_dir = Path(weights_dir)
        if torch.cuda.device_count() > 1:
            print(f"Using {torch.cuda.device_count()} GPUs with DataParallel")
            self.model = nn.DataParallel(model).to(device)
        else:
            self.model = model.to(device)
            
        self.train_loader = train_loader
        self.val_loader = val_loader
        self.scaler = GradScaler()
        base_model = self.model.module if hasattr(self.model, 'module') else self.model
        base_model.head.alpha = torch.tensor([1.0, 1.0, 2.5], device=device)
        self.criterion = base_model.head.compute_loss
        self.best_val_loss = float('inf')

    def train_epoch(self, optimizer, epoch):
        self.model.train()
        pbar = tqdm(self.train_loader, desc=f"Epoch {epoch}")
        total_loss = 0
        for batch in pbar:
            imgs = batch['img'].to(self.device).float() / 255.0
            optimizer.zero_grad()
            with autocast():
                preds = self.model(imgs)
                loss = self.criterion(preds, batch, self.device)
            self.scaler.scale(loss).backward()
            self.scaler.step(optimizer)
            self.scaler.update()
            total_loss += loss.item()
            pbar.set_postfix({"loss": f"{loss.item():.4f}"})
        return total_loss / len(self.train_loader)

    def validate(self):
        self.model.eval()
        val_loss = 0
        with torch.no_grad():
            for batch in self.val_loader:
                imgs = batch['img'].to(self.device).float() / 255.0
                with autocast():
                    preds = self.model(imgs)
                    loss = self.criterion(preds, batch, self.device)
                val_loss += loss.item()
        return val_loss / len(self.val_loader)

    def run(self, epochs=50):
        base_model = self.model.module if hasattr(self.model, 'module') else self.model
        base_model.backbone.freeze()
        optimizer = optim.AdamW(filter(lambda p: p.requires_grad, self.model.parameters()), lr=1e-4)
        
        resume_path = self.weights_dir / "last.pt"
        start_epoch = 0
        if resume_path.exists():
            print(f"Resuming from {resume_path}")
            checkpoint = torch.load(resume_path)
            base_model.load_state_dict(checkpoint['model_state_dict'])
            optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
            start_epoch = checkpoint['epoch']

        for epoch in range(start_epoch + 1, epochs + 1):
            if epoch == 10:
                print("\n[INFO] Unfreezing backbone for fine-tuning...")
                base_model.backbone.unfreeze()
                optimizer = optim.AdamW(self.model.parameters(), lr=1e-5)
            
            avg_loss = self.train_epoch(optimizer, epoch)
            val_loss = self.validate()
            print(f"Epoch {epoch} | Train Loss: {avg_loss:.4f} | Val Loss: {val_loss:.4f}")
            
            if epoch % 2 == 0:
                torch.save({'epoch': epoch, 'model_state_dict': base_model.state_dict(), 'optimizer_state_dict': optimizer.state_dict()}, self.weights_dir / "last.pt")
            
            if val_loss < self.best_val_loss:
                self.best_val_loss = val_loss
                torch.save(base_model.state_dict(), self.weights_dir / "best.pt")
                print(f"--> Best model saved with val_loss: {val_loss:.4f}")

### 4. Production Initiation

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Target Hardware: {device}")

# We provide a full writable path for the backbone variant to avoid Read-Only errors
BACKBONE_PATH = "/kaggle/working/yolo11m.pt"

model = HybridWeaponDetector(
    backbone_variant=BACKBONE_PATH, 
    pretrained=True, 
    nc=3, 
    device=device
)

if DATA_YAML.exists():
    train_loader, val_loader = get_dataloaders(DATA_YAML, DATA_ROOT, batch_size=32)
    trainer = HybridTrainer(model, train_loader, val_loader, device=device, weights_dir=WEIGHTS_DIR)
    trainer.run(epochs=50)
else:
    print(f"[ERROR] data.yaml not found. Checked: {DATA_YAML}")